# 01 — Data Access Test (Checkpoint 1)

**Goal:** load **one small real** NOAA POES/MetOp SEM‑2 file, print its metadata, list its
variables, and show a few rows/values — confirming the MVP fields exist
(*time, latitude, longitude, satellite ID, at least one proton channel*).

**No maps and no scientific conclusions here** (that is Checkpoint 2).

> **Honesty note.** This notebook was *authored* on 2026‑06‑07 but **not executed** in the
> authoring environment (Python 3.14, no `pip`/`xarray`/Jupyter). It therefore ships with
> **no saved outputs**. **Path A** (ASCII) needs only the standard library and was verified
> independently via the shell; **Path B** (NetCDF) needs `pip install xarray netCDF4`.
> Source URLs and checksums are in `../data/samples/PROVENANCE.md`.

## Path A — ASCII Level‑2 (zero dependencies, runs anywhere)
Self‑describing header; `sslat`/`sslon` = sub‑satellite geographic lat/lon; `mep0p6` / `mepomp6`
= high‑energy proton count rates (counts/s, uncorrected).

In [ ]:
import os, urllib.request

TXT_URL = "https://www.ncei.noaa.gov/data/poes-metop-space-environment-monitor/access/l2/v01r00/txt/2014/noaa19/poes_n19_20140102.txt"
LOCAL = os.path.join("..", "data", "samples", "poes_n19_20140102.txt")

if not os.path.exists(LOCAL):
    os.makedirs(os.path.dirname(LOCAL), exist_ok=True)
    print("Downloading", TXT_URL)
    urllib.request.urlretrieve(TXT_URL, LOCAL)
print("Using local file:", LOCAL, "(", os.path.getsize(LOCAL), "bytes )")

with open(LOCAL) as f:
    lines = f.read().splitlines()
header = lines[0].split()
rows = [ln.split() for ln in lines[1:] if ln.strip()]
ci = {c: i for i, c in enumerate(header)}

print("columns (%d):" % len(header))
print(" ", " ".join(header))
print("data rows:", len(rows))

def col(name):
    j = ci[name]
    return [float(r[j]) for r in rows if len(r) > j]

sslat, sslon = col("sslat"), col("sslon")
print("sslat range: %.2f .. %.2f deg" % (min(sslat), max(sslat)))
print("sslon range: %.2f .. %.2f deg" % (min(sslon), max(sslon)))
print("mep0p6  (0deg proton P6, counts/s): %.1f .. %.1f" % (min(col("mep0p6")),  max(col("mep0p6"))))
print("mepomp6 (omni dome P6,  counts/s): %.1f .. %.1f" % (min(col("mepomp6")), max(col("mepomp6"))))

print()
print("first 3 records (time + position + proton channel):")
for r in rows[:3]:
    print("  %s-%s-%s %s:%s:%s  sslat=%s sslon=%s  mep0p6=%s mepomp6=%s" % (
        r[ci['year']], r[ci['mo']], r[ci['dy']], r[ci['hr']], r[ci['mi']], r[ci['second']],
        r[ci['sslat']], r[ci['sslon']], r[ci['mep0p6']], r[ci['mepomp6']]))

## Path B — NetCDF Level‑1b (science‑ready; requires `pip install xarray netCDF4`)
Calibrated flux + full magnetic ephemeris. `lat`/`lon` = geographic; `mep_omni_flux_p1` =
integral omnidirectional high‑energy proton flux (the recommended first SAA channel).

In [ ]:
import os

NC_URL = "https://www.ncei.noaa.gov/data/poes-metop-space-environment-monitor/access/l1b/v01r00/2024/noaa19/poes_n19_20240101_proc.nc"
LOCAL_NC = os.path.join("..", "data", "samples", "poes_n19_20240101_proc.nc")

if not os.path.exists(LOCAL_NC):
    import urllib.request
    os.makedirs(os.path.dirname(LOCAL_NC), exist_ok=True)
    print("Downloading", NC_URL)
    urllib.request.urlretrieve(NC_URL, LOCAL_NC)

try:
    import xarray as xr
except ImportError:
    raise SystemExit("xarray/netCDF4 not installed. Run: pip install xarray netCDF4")

ds = xr.open_dataset(LOCAL_NC)
print("dimensions:", dict(ds.sizes))
print("number of variables:", len(ds.variables))

key = ["time", "year", "day", "msec", "lat", "lon", "alt", "satID", "sat_direction",
       "mep_pro_tel0_flux_p1", "mep_pro_tel0_flux_p6", "mep_pro_tel90_flux_p1",
       "mep_omni_flux_p1", "mep_omni_flux_p2", "mep_omni_flux_p3",
       "mep_ele_tel0_flux_e1", "L_IGRF", "MLT", "mag_lat_sat", "mep_IFC_on"]
print()
print("key variables present:")
for v in key:
    if v in ds.variables:
        da = ds[v]
        print("  %-22s dims=%-14s units=%s" % (v, str(da.dims), da.attrs.get("units", "-")))
    else:
        print("  %-22s (NOT FOUND)" % v)

print()
print("example values (first 3 records):")
d0 = list(ds.sizes)[0]
for v in ["lat", "lon", "mep_omni_flux_p1"]:
    if v in ds.variables:
        print("  %-18s %s" % (v, ds[v].isel({d0: slice(0, 3)}).values))
ds.close()

## What this confirms
Both real files contain **time, latitude, longitude, satellite identity, and proton channels** —
the MVP fields. **Next (Checkpoint 2):** tidy loader → lon×lat binning → first exploratory SAA
footprint of `mep_omni_flux_p1` (filtering `mep_IFC_on`). No conclusions drawn here.